# Fatbox — Fault extraction from TanDEM-X DEM (Afar region)

Adapted from `Tuto_DEM_GLO30_notebook.ipynb` for the local TanDEM-X DEM in UTM Zone 37.

**Input file:** `TDM1_DEM_30m_UTM37.tif`  
**Resolution:** 30 m  
**DEM size:** 7343 × 11073 pixels (~222 km × 334 km)  
**Projection:** UTM Zone 37 (EPSG:32637)  
**NW corner (UTM):** X = 716 922 m, Y = 1 440 548 m

Workflow:
1. Import — load a tile of the DEM
2. Mapping — edge detection → skeleton
3. Classification — build a graph from the skeleton
4. Filtering — clean the network
5. Border processing — prepare for structural analysis
6. Structural analysis — strike, throw, extension, displacement
7. Plots — visualise the results

## Imports

In [ ]:
import math
from matplotlib import cm
import matplotlib as mpl
import matplotlib.pyplot as plt
plt.close('all')
import numpy as np
from skimage.morphology import skeletonize
import time
import pickle
import statistics
from skimage import feature, morphology, filters
import cv2
import networkx as nx
from scipy.spatial import distance_matrix
import sys
import os
import earthpy
import earthpy.spatial
import earthpy.plot
from pathlib import Path

# ── Fatbox modules ──────────────────────────────────────────────────────────
path_folder = Path('/home/guiguizz/stage/modules')
os.chdir(path_folder)

import preprocessing
import metrics
import plots
import utils
import structural_analysis
import edits

print('All imports OK')

## Path management & output directories

In [ ]:
# ── Input DEM ───────────────────────────────────────────────────────────────
DEM_path = Path('/home/guiguizz/stage/TDM1_DEM_30m_UTM37.tif')

# ── Output root ─────────────────────────────────────────────────────────────
save_path = Path('/home/guiguizz/stage/tutorials/digital_elevation_models')

save_plots = True
my_dpi = 400

# ── Create output directories ────────────────────────────────────────────────
for sub in ['array/extracted', 'array/analyzed', 'images/extracted', 'images/analyzed']:
    (save_path / sub).mkdir(parents=True, exist_ok=True)

loc_map     = save_path / 'images' / 'extracted'
loc_analyse = save_path / 'images' / 'analyzed'

print('Directories ready.')

## 1 — Load DEM tile

The full DEM is 7343 × 11073 pixels.  
Coordinates below are **pixel indices** (row / column, 0-based, top-left origin).

| Pixel coordinate | UTM Easting (m) | UTM Northing (m) | Approx. geographic |
|---|---|---|---|
| col 0 | 716 922 | — | ~39.4 °E |
| col 7342 | 938 665 | — | ~41.5 °E |
| row 0 | — | 1 440 548 | ~13.0 °N |
| row 11072 | — | 1 106 028 | ~10.0 °N |

**How to pick your tile:**  
- To cover a 30 km × 30 km area start with a 1000 × 1000 pixel window.  
- To process the full DEM set `coord_south = 11073` and `coord_east = 7343`  
  (expect several minutes of runtime on large tiles).

Alternatively you can supply **geographic coordinates** (decimal degrees) by  
changing `system='geographic'` and replacing the coord values with lat/lon.

In [ ]:
# ── Tile selection (pixel indices) ──────────────────────────────────────────
# Row 0 = northern edge, Row 11072 = southern edge
# Col 0 = western edge,  Col 7342  = eastern edge

coord_north = 0      # first row to include  (adjust to your area of interest)
coord_south = 1000   # last  row  (exclusive)
coord_west  = 0      # first column
coord_east  = 1000   # last  column (exclusive)

# For the full DEM:
# coord_north = 0; coord_south = 11073; coord_west = 0; coord_east = 7343

img_dem, lat, lon = preprocessing.extract_geotiff_and_coord(
    str(DEM_path),
    coord_north=coord_north,
    coord_south=coord_south,
    coord_west=coord_west,
    coord_east=coord_east,
    system='grid coordinates'
)

sy_dem, sx_dem = img_dem.shape
resolution = 30  # horizontal resolution in metres (TanDEM-X 30 m)

print(f'Loaded tile: {sy_dem} rows × {sx_dem} cols')
print(f'Coverage: ~{sy_dem * resolution / 1000:.1f} km (N–S) × {sx_dem * resolution / 1000:.1f} km (E–W)')
print(f'Elevation range: {img_dem.min():.0f} m – {img_dem.max():.0f} m')

# Optional: save extracted arrays so you only extract once
# np.save(save_path / 'array' / 'extracted' / 'img_dem_afar.npy', img_dem)
# np.save(save_path / 'array' / 'extracted' / 'lat_afar.npy',     lat)
# np.save(save_path / 'array' / 'extracted' / 'lon_afar.npy',     lon)

In [ ]:
# Quick DEM preview
fig, ax = plt.subplots(figsize=(8, 8))
a = ax.imshow(img_dem, cmap='gray_r')
ax.set_title('DEM — Afar region (TanDEM-X 30 m)')
c = plt.colorbar(a)
c.set_label('Elevation [m]')
if save_plots:
    plt.savefig(loc_map / 'DEM_afar.png', dpi=my_dpi)
plt.show()

## 2 — Fault extraction (mapping)

Three successive image-processing steps:
1. **Gaussian smoothing** (`s_smoothed`) — suppresses pixel noise before edge detection.
2. **Canny edge detection** (`s_edges`) — detects topographic discontinuities (fault scarps).
3. **Cleaning** (`msize_cleaned`) — removes small spurious detections.
4. **Skeletonisation** — thins edges to 1-pixel-wide lines.

**Tuning tips for the Afar DEM:**
- Increase `s_smoothed` (e.g. 2–3) if the result is very noisy (lava textures, wadis).
- Increase `s_edges` to focus on large, clear scarps; decrease for finer features.
- Increase `msize_cleaned` to remove short artefacts (value in pixels; 30 px = 900 m).

In [ ]:
# ── Extraction parameters — adjust these !! ─────────────────────────────────
s_smoothed   = 1    # Gaussian σ for pre-smoothing (0 = none)
s_edges      = 3    # Canny σ for edge detection
con_cleaned  = 2    # connectivity for noise removal (leave at 2)
msize_cleaned = 30  # minimum object size in pixels (30 px ≈ 900 m)

# ── Processing ──────────────────────────────────────────────────────────────
smoothed = filters.gaussian(img_dem, sigma=s_smoothed)
edges    = feature.canny(smoothed, sigma=s_edges)
cleaned  = morphology.remove_small_objects(edges, connectivity=con_cleaned, min_size=msize_cleaned)
skeleton = skeletonize(cleaned)
ret, markers = cv2.connectedComponents(skeleton.astype('uint8'))

print(f'Connected components detected: {ret - 1}')

# ── Optional save ───────────────────────────────────────────────────────────
save_extraction = False
if save_extraction:
    loc = save_path / 'array' / 'extracted'
    np.save(loc / f'smoothed_sigma={s_smoothed}', smoothed)
    np.save(loc / f'edges_sigma{s_edges}',        edges)
    np.save(loc / f'cleaned_con={con_cleaned}_min_s={msize_cleaned}', cleaned)
    np.save(loc / 'skeleton',                     skeleton)
    np.save(loc / f'markers_ret={ret}',            markers)

In [ ]:
hillshade = earthpy.spatial.hillshade(img_dem)

fig, axs = plt.subplots(2, 3, figsize=(16, 11))

axs[0,0].imshow(img_dem, cmap='gray_r')
axs[0,0].set_title('DEM')

axs[0,1].imshow(smoothed, cmap='gray_r')
axs[0,1].set_title(f'Smoothed σ={s_smoothed}')

axs[0,2].imshow(edges, cmap='gray_r')
axs[0,2].set_title(f'Edges σ={s_edges}')

axs[1,0].imshow(cleaned, cmap='gray_r')
axs[1,0].set_title(f'Cleaned (min size={msize_cleaned})')

axs[1,1].imshow(skeleton, cmap='gray_r')
axs[1,1].set_title('Skeleton')

# Edges overlaid on hillshade
earthpy.plot.plot_bands(hillshade, cbar=False, ax=axs[1,2], alpha=0.6)
axs[1,2].contour(edges, cmap='Paired_r', linewidths=0.7, alpha=0.9)
axs[1,2].set_title('Edges on hillshade')

plt.tight_layout()
if save_plots:
    plt.savefig(loc_map / 'Extraction_overview.png', dpi=my_dpi)
plt.show()

## 3 — Classification (build fault network graph)

Each pixel of the skeleton becomes a graph node. Nearby nodes belonging to the  
same connected component are linked by edges — each component represents one fault.

In [ ]:
G = nx.Graph()

node = 0
for comp in range(1, ret):
    points = np.transpose(np.vstack(np.where(markers == comp)))
    for point in points:
        G.add_node(node)
        G.nodes[node]['pos']       = (point[1], point[0])
        G.nodes[node]['component'] = comp
        G.nodes[node]['polarity']  = 0
        node += 1

# ── Distance threshold !! ───────────────────────────────────────────────────
distance_threshold = 1.5  # pixels — connect nodes closer than this

for comp in range(1, ret):
    points = [G.nodes[n]['pos'] for n in G if G.nodes[n]['component'] == comp]
    nodes  = [n for n in G if G.nodes[n]['component'] == comp]
    dm     = distance_matrix(points, points)
    for n in range(len(points)):
        for m in range(len(points)):
            if dm[n, m] < distance_threshold:
                G.add_edge(nodes[n], nodes[m])

G = edits.label_components(G)
G = edits.remove_self_edge(G)

print(f'Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

# ── Optional save / load ─────────────────────────────────────────────────────
save_G = False
if save_G:
    loc = save_path / 'array' / 'extracted'
    pickle.dump(G, open(loc / 'G_afar.pickle', 'wb'))

load_G = False
if load_G:
    loc = save_path / 'array' / 'extracted'
    G   = pickle.load(open(loc / 'G_afar.pickle', 'rb'))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(img_dem, cmap=cm.gray_r)
plots.plot_components(G, True, ax=ax, node_size=0.8)
ax.set_title('Raw fault network')
plt.tight_layout()
if save_plots:
    plt.savefig(loc_map / 'G_afar_raw.png', dpi=my_dpi)
plt.show()

## 4 — Filter the network

Chain of edits to clean the raw graph:
- **`connect_close_components`** — merge endpoints that are close but not yet connected.
- **`simplify`** — reduce node density (keeps 1 node every N nodes).
- **`split_triple_junctions`** — separate forked fault tips.
- **`loop_cut_U_global`** — remove U-shaped artefacts using the DEM to decide which side is the scarp.
- **`remove_small_components`** — discard components with fewer than N nodes.

In [ ]:
G_filtered_1 = edits.connect_close_components(G, 4.0)
G_filtered_1 = edits.label_components(G_filtered_1)

G_filtered_2 = edits.simplify(G_filtered_1, 3)
G_filtered_2 = edits.split_triple_junctions(G_filtered_2, 6)
G_filtered_2 = edits.remove_node_alone(G_filtered_2)
G_filtered_2 = edits.label_components(G_filtered_2)

G_filtered_3 = edits.loop_cut_U_global(G_filtered_2, img_dem)
G_filtered_3 = edits.label_components(G_filtered_3)
G_filtered_3 = edits.remove_small_components(G_filtered_3, 5)

print(f'After filtering: {G_filtered_3.number_of_nodes()} nodes, {G_filtered_3.number_of_edges()} edges')

save_filtered = False
if save_filtered:
    loc = save_path / 'array' / 'extracted'
    pickle.dump(G_filtered_3, open(loc / 'G_afar_filtered.pickle', 'wb'))

load_filtered = False
if load_filtered:
    loc        = save_path / 'array' / 'extracted'
    G_filtered_3 = pickle.load(open(loc / 'G_afar_filtered.pickle', 'rb'))

## 5 — Border processing

Remove edges whose cross-section would fall outside the DEM array  
(avoids index-out-of-bounds errors during structural analysis).  
Parameter `d` = half-length of the cross-section in pixels.

> **d = 12 px × 30 m = 360 m on each side.** Increase if your faults are large and  
> you want a wider topographic profile; decrease if you are close to the DEM edge.

In [ ]:
J = G_filtered_3.copy()

J = structural_analysis.strike_edges(J)
J = metrics.compute_edge_length_meters(J, resolution)
J = metrics.compute_edge_length(J)

d = 12  # cross-section half-length in pixels  !! adjust if needed
J = structural_analysis.filter_out_edges(J, img_dem, d=d)

J = edits.label_components(J)
J = edits.remove_small_components(J, 5)
J = edits.label_components(J)

print(f'After border processing: {J.number_of_nodes()} nodes, {J.number_of_edges()} edges')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))
hillshade = earthpy.spatial.hillshade(img_dem)
ax.imshow(img_dem, cmap='gray_r')
earthpy.plot.plot_bands(hillshade, cbar=False, ax=ax, alpha=0.4)
plots.plot_components(J, label=True, node_size=5, ax=ax)
ax.set_title('Filtered fault network (labelled)')
plt.tight_layout()
if save_plots:
    plt.savefig(loc_map / 'G_afar_filtered.png', dpi=my_dpi)
plt.show()

## 6 — Structural analysis

For each fault segment a topographic **cross-section** is drawn perpendicular to the fault,  
extending `d` pixels on each side of the segment midpoint. From the elevation profile:

- **Throw** — maximum vertical offset across the scarp (metres)
- **Natural dip** — apparent dip angle from the throw and half-width
- **Extension** — horizontal component of the displacement (uses constrained dip if selected)
- **Displacement** — total slip vector magnitude

`summary_matrix` columns:  
`[label, mean_strike°, total_length_m, max_dip°, mean_throw_m, mean_extension_m, max_displacement_m]`

In [ ]:
# ── Settings !! ─────────────────────────────────────────────────────────────
d             = 12    # cross-section half-length in pixels (must match border step)
pix_size      = resolution
dip_constrain = 60   # degrees — used when use_natural_dip=False
use_natural_dip = False  # True: use measured dip; False: use constrained dip

# ── Loop over all fault components ──────────────────────────────────────────
list_labels    = metrics.get_component_labels(J)
summary_matrix = np.zeros((len(list_labels), 7))
extremities    = np.zeros((len(list(J.edges)), 4))

k = 0
i = 0
list_edge    = []
list_dip_all = []

for label in list_labels:
    nodes = [node for node in J if J.nodes[node]['component'] == label]
    K     = nx.subgraph(J, nodes)

    if len(list_labels) <= 200 and (i in np.arange(10, 210, 10)):
        print(i)
    elif len(list_labels) > 200 and (i in np.arange(200, max(list_labels), 50)):
        print(i)

    summary_matrix[i, 0] = label
    length         = 0
    list_extension = []
    list_throw     = []
    list_disp      = []
    list_strike    = []
    list_dip_label = []

    for edge in K.edges:
        list_edge.append(edge)
        list_th          = []
        list_kmax_throw  = []

        x0, y0 = K.nodes[edge[0]]['pos']
        x1, y1 = K.nodes[edge[1]]['pos']
        strike  = K.edges[edge]['strike']

        all_x = np.zeros([2 * d + 1, 1])
        all_y = np.zeros([2 * d + 1, 1])
        alt   = np.zeros([2 * d + 1, 1])

        midpoint_x  = (x0 + x1) / 2
        midpoint_y  = (y0 + y1) / 2
        midpoint_xx = math.floor(midpoint_x + 0.5)
        midpoint_yy = math.floor(midpoint_y + 0.5)

        all_x[d, 0] = midpoint_x
        all_y[d, 0] = midpoint_y
        alt[d, 0]   = img_dem[midpoint_yy, midpoint_xx]

        list_dip_edge = []

        for n in range(1, d + 1):
            xD, yD, xE, yE = structural_analysis.calcul_coordinates(
                np.radians(strike), midpoint_x, midpoint_y, n
            )
            all_x[d - n, 0] = xD
            all_y[d - n, 0] = yD
            all_x[d + n, 0] = xE
            all_y[d + n, 0] = yE

            xD = math.floor(xD + 0.5)
            yD = math.floor(yD + 0.5)
            xE = math.floor(xE + 0.5)
            yE = math.floor(yE + 0.5)

            alt[d - n, 0] = img_dem[yD, xD]
            alt[d + n, 0] = img_dem[yE, xE]

            throw    = img_dem[yD, xD] - img_dem[yE, xE]
            list_th.append(abs(throw))

            dip_rad = np.arctan(throw / (2 * n * pix_size))
            list_dip_edge.append(np.degrees(dip_rad))

        xD, yD, xE, yE = structural_analysis.calcul_coordinates(
            np.radians(strike), midpoint_x, midpoint_y, d
        )
        extremities[k, :] = [xD, yD, xE, yE]

        kmax  = list_th.index(max(list_th)) + 1
        list_kmax_throw.append(kmax)
        throw = abs(max(list_th))

        dip_dg = np.degrees(np.arctan(throw / (2 * n * pix_size)))

        list_dip_all.append(max(list_dip_edge))
        list_dip_label.append(max(list_dip_edge))

        dip = dip_constrain if not use_natural_dip else dip_dg

        extension    = throw / np.tan(np.radians(abs(dip)))
        displacement = np.sqrt(throw ** 2 + extension ** 2)

        J.edges[edge]['extension']    = extension
        J.edges[edge]['throw']        = throw
        J.edges[edge]['displacement'] = displacement
        if use_natural_dip:
            J.edges[edge]['natural_dip'] = dip_dg

        list_strike.append(strike)
        list_extension.append(extension)
        list_throw.append(throw)
        list_disp.append(displacement)
        length += J.edges[edge]['length']

        xD_i = math.floor(xD + 0.5)
        yD_i = math.floor(yD + 0.5)
        xE_i = math.floor(xE + 0.5)
        yE_i = math.floor(yE + 0.5)
        J.edges[edge]['dip_direction'] = 'west' if img_dem[yD_i, xD_i] < img_dem[yE_i, xE_i] else 'east'

        k += 1

    summary_matrix[i, 1] = np.mean(list_strike)
    summary_matrix[i, 2] = length
    summary_matrix[i, 3] = max(list_dip_label)
    summary_matrix[i, 4] = np.mean(list_throw)
    summary_matrix[i, 5] = np.mean(list_extension)
    summary_matrix[i, 6] = max(list_disp)

    i += 1

print('Structural analysis complete.')
print(f'summary_matrix shape: {summary_matrix.shape}')
# Columns: [label, mean_strike°, total_length_m, max_dip°, mean_throw_m, mean_extension_m, max_displacement_m]

In [ ]:
save_analysis = False
if save_analysis:
    loc = save_path / 'array' / 'analyzed'
    pickle.dump(J, open(loc / 'J_afar_analyzed.pickle', 'wb'))
    np.save(loc / 'summary_matrix_afar', summary_matrix)

load_analysis = False
if load_analysis:
    loc            = save_path / 'array' / 'analyzed'
    J              = pickle.load(open(loc / 'J_afar_analyzed.pickle', 'rb'))
    summary_matrix = np.load(loc / 'summary_matrix_afar.npy')

## 7 — Plots

In [ ]:
# Rose diagram of fault strikes
ax = plots.plot_rose_diagram(J)
ax.figure.suptitle('Strike rose diagram — Afar')
if save_plots:
    ax.figure.savefig(loc_analyse / 'Rose_plot_afar.png', dpi=my_dpi)

In [ ]:
# Length distribution
list_labels = metrics.get_component_labels(J)
fig1, fig2  = plots.plot_length_histogram(J, list_labels, resolution=resolution, title='Length histogram — Afar')
if save_plots:
    fig1.savefig(loc_analyse / 'Length_histogram_bars_afar.png', dpi=my_dpi)
    fig2.savefig(loc_analyse / 'Length_histogram_dots_afar.png', dpi=my_dpi)

In [ ]:
# Extension map overlaid on DEM
fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(img_dem, cmap=cm.gray_r)
plots.plot_edge_attribute_other_version(J, 'extension', ax=ax)
ax.set_title('Extension [m] — Afar')
if save_plots:
    plt.savefig(loc_analyse / 'edge_extension_afar.png', dpi=my_dpi)
plt.show()

In [ ]:
# Throw map
fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(img_dem, cmap=cm.gray_r)
plots.plot_edge_attribute_other_version(J, 'throw', ax=ax)
ax.set_title('Throw [m] — Afar')
if save_plots:
    plt.savefig(loc_analyse / 'edge_throw_afar.png', dpi=my_dpi)
plt.show()

In [ ]:
# Displacement–Length scatter for the whole network
list_labels = metrics.get_component_labels(J)
disp_length = np.zeros((len(list_labels), 2))

for ii, label in enumerate(list_labels):
    nodes = [n for n in J if J.nodes[n]['component'] == label]
    K     = nx.subgraph(J, nodes)
    disps  = [K.edges[e]['throw'] for e in K.edges]
    lengths = sum(K.edges[e]['length'] * resolution for e in K.edges)
    disp_length[ii, 0] = lengths
    disp_length[ii, 1] = max(disps)

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(disp_length[:, 0], disp_length[:, 1],
           c=np.random.rand(len(disp_length), 3), s=20)
ax.set_xlabel('Fault length [m]')
ax.set_ylabel('Maximum throw [m]')
ax.set_title('Displacement–Length — Afar')
if save_plots:
    plt.savefig(loc_analyse / 'DL_scatter_afar.png', dpi=my_dpi)
plt.show()

In [ ]:
# Displacement profile along a single fault
# Set label to the component number you want to inspect (visible on the labelled map above)

label = list_labels[0]  # !! change to your label of interest

nodes = [n for n in J if J.nodes[n]['component'] == label]
K     = nx.subgraph(J, nodes)
k     = K.number_of_edges()

DL_matrix = np.zeros((k, 2))
endpoints = [n for n in K if K.degree[n] == 1]

if len(endpoints) >= 2:
    path_list = list(nx.shortest_simple_paths(K, source=endpoints[0], target=endpoints[1]))[0]
    length = 0
    for idx, l in enumerate(range(len(path_list) - 1)):
        edge   = (path_list[l], path_list[l + 1])
        length += K.edges[edge]['length'] * resolution
        DL_matrix[idx, 0] = K.edges[edge]['displacement']
        DL_matrix[idx, 1] = length

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(DL_matrix[:idx + 1, 1], DL_matrix[:idx + 1, 0])
    ax.set_xlabel('Distance along fault [m]')
    ax.set_ylabel('Displacement [m]')
    ax.set_title(f'D/L profile — label {label}')
    if save_plots:
        plt.savefig(loc_analyse / f'DL_label{label}_afar.png', dpi=my_dpi)
    plt.show()
else:
    print(f'Component {label} has fewer than 2 endpoints — cannot compute D/L profile.')